# MMAR Simulation — Google Colab

[mmar-sim](https://github.com/Rifur/mmar-sim)

**主要輸出是文字報告**（百分位價格、機率表、回檔／現價情境、VaR、GOF 檢定）— 執行後請在 cell **輸出區向上捲動** 閱讀完整 `print` 報告；圖表顯示在報告之後。

1. **Runtime → Run all**
2. **Step 2** 改 `TICKER` / `N_STEPS` / `N_SIMS` 後執行
3. 或用 **Step 3** 表單輸入股票代號

範例：`2330.TW`、`6919.TW`、`NVDA`、`0050.TW`

## Step 1 — Setup

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/Rifur/mmar-sim.git"


def _ensure_repo() -> None:
    if os.path.isfile("real_fractal_sim.py"):
        return
    if os.path.isdir("mmar-sim") and os.path.isfile("mmar-sim/real_fractal_sim.py"):
        os.chdir("mmar-sim")
        return
    print("Cloning mmar-sim from GitHub...")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL])
    os.chdir("mmar-sim")


_ensure_repo()
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "yfinance>=1.4", "matplotlib>=3.8", "pandas>=2.0",
    "scipy>=1.11", "curl-cffi>=0.15",
])
print(f"Ready: {os.getcwd()}")

## Step 2 — 輸入股票代號，顯示完整報告

`run_colab()` 會 `print` 完整分析報告（與本機 CLI 相同）。修改下面三行後執行此 cell。

In [ ]:
%matplotlib inline
from real_fractal_sim_colab import run_colab, print_report

TICKER = "6919.TW"   # 2330.TW, NVDA, 0050.TW ...
N_STEPS = 20         # 模擬天數（交易日）
N_SIMS = 10000       # 路徑數

# 完整文字報告 + 圖表（報告在 output 區，請向上捲動）
result = run_colab(TICKER, n_steps=N_STEPS, n_sims=N_SIMS, seed=42)

## Step 2b — 重新顯示報告（不重跑模擬）

若 output 被圖表推太下面，可再執行此 cell 重印文字報告。

In [ ]:
from real_fractal_sim_colab import print_report

print_report(result)

## Step 3 — 表單輸入（可選）

輸入股票代號，點 **Run MMAR**。完整報告顯示在下方捲動區。

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from real_fractal_sim_colab import run_colab

ticker_in = widgets.Text(value="2330.TW", description="Ticker", layout=widgets.Layout(width="280px"))
steps_in = widgets.IntSlider(value=252, min=5, max=504, step=1, description="Days", readout_format="d")
sims_in = widgets.IntSlider(value=5000, min=500, max=20000, step=500, description="Paths", readout_format="d")
run_btn = widgets.Button(description="Run MMAR", button_style="success", icon="play")
report_out = widgets.Output(
    layout=widgets.Layout(width="100%", height="640px", border="1px solid #ccc", overflow_y="auto")
)


def _on_run(_):
    with report_out:
        clear_output(wait=True)
        r = run_colab(
            ticker_in.value.strip().upper(),
            n_steps=steps_in.value,
            n_sims=sims_in.value,
            seed=42,
        )
        globals()["result"] = r


run_btn.on_click(_on_run)
display(widgets.VBox([ticker_in, steps_in, sims_in, run_btn, report_out]))

## Step 4 — 下載圖表

In [ ]:
try:
    from google.colab import files
    files.download(result["mmar_path"])
    files.download(result["gof_path"])
except (ImportError, NameError, KeyError):
    print("請先執行 Step 2 或 Step 3。")

## 一行程式碼

```python
from real_fractal_sim_colab import run_colab
result = run_colab("6919.TW", n_steps=20, n_sims=10000)
# 完整報告已 print；只看分數：
print(result["fit"]["score"])

# 重印報告：
from real_fractal_sim_colab import print_report
print_report(result)
```

只要文字、不要圖：`run_colab("6919.TW", n_steps=20, show_plots=False)`